# Welcome to Learning Segmentation with Khadijah

Foundations for Image Segmentation
===========================================

This files has code to cover the
basic skills you need *before* tackling a full‑blown U‑Net.

Sections
--------
1.  Loading an image & NumPy tensor basics
2.  Thinking in [C, H, W] – PyTorch tensor conversions
3.  Manual 3×3 convolution on a toy image (stride 1, padding 1)
4.  Tiny 2‑layer MLP that trains on MNIST (CPU‑friendly)


# 0. Imports – the absolute minimum we need


In [8]:
from pathlib import Path
import hashlib
import urllib.request

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm


# 1. Loading a sample image & NumPy basics


In [10]:

print("[1] Loading sample image and thresholding the red channel …")
IMG_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
IMG_FNAME = Path("sample_dog.jpg")

if not IMG_FNAME.exists():
    print("  downloading sample image …")
    urllib.request.urlretrieve(IMG_URL, IMG_FNAME)

# Quick integrity check – we *warn* instead of crashing.
md5_short = hashlib.md5(IMG_FNAME.read_bytes()).hexdigest()[:6]
if md5_short != "3e27a4":
    print(f"  Checksum mismatch (expected 3e27a4, got {md5_short}). Continuing anyway …")

img_pil = Image.open(IMG_FNAME).convert("RGB")
img_np = np.asarray(img_pil)

# Threshold red channel: any pixel with R > 200 ➜ white (255,255,255)
red_channel = img_np[:, :, 0]
mask = red_channel > 200
img_np_thresh = img_np.copy()
img_np_thresh[mask] = 255

# Convert back to PIL and save preview
Image.fromarray(img_np_thresh).save("threshold_preview.jpg")
print("   Saved 'threshold_preview.jpg' (red‑thresholded image)\n")


[1] Loading sample image and thresholding the red channel …
  Checksum mismatch (expected 3e27a4, got dd0a67). Continuing anyway …
   Saved 'threshold_preview.jpg' (red‑thresholded image)



# 2. Converting between PIL, NumPy, and Torch tensors

In [11]:

print("[2] Verifying PIL ↔ NumPy ↔ Torch round‑trip …")

# PIL → Torch tensor (C, H, W) and normalise to [0,1]
img_tensor = transforms.ToTensor()(img_pil)
assert img_tensor.shape[0] == 3, "Tensor should have 3 colour channels"

# Torch → NumPy → PIL
img_np_round = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
img_pil_round = Image.fromarray(img_np_round)

# Check pixels identical
assert np.array_equal(np.asarray(img_pil), img_np_round), "Round‑trip altered pixels!"
print("   Round‑trip lossless\n")

[2] Verifying PIL ↔ NumPy ↔ Torch round‑trip …
   Round‑trip lossless




# 3. Manual 3×3 convolution on a toy image

In [13]:

print("[3] Manual 3×3 mean filter on a 5×5 toy image …")

toy = np.arange(25).reshape(5, 5).astype(np.float32)
# Pad with 1‑pixel border so output size matches input
padded = np.pad(toy, 1, mode="constant", constant_values=0)

kernel = np.ones((3, 3), dtype=np.float32) / 9.0
out = np.zeros_like(toy)

for i in range(5):
    for j in range(5):
        window = padded[i : i + 3, j : j + 3]
        out[i, j] = np.sum(window * kernel)

print("  input:\n", toy)
print("  output (mean‑blur):\n", out)
print("  Manual conv complete\n")


[3] Manual 3×3 mean filter on a 5×5 toy image …
  input:
 [[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]
 [15. 16. 17. 18. 19.]
 [20. 21. 22. 23. 24.]]
  output (mean‑blur):
 [[ 1.3333334  2.3333335  3.         3.666667   2.6666665]
 [ 3.666667   6.0000005  7.         8.         5.666667 ]
 [ 7.        11.        12.        13.         9.       ]
 [10.333333  16.        17.        18.        12.333334 ]
 [ 8.        12.333333  13.        13.666667   9.333334 ]]
  Manual conv complete



# 4. Tiny 2‑layer MLP on MNIST

In [17]:

print("[4] Training a 2‑layer MLP on MNIST")

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
train_ds = datasets.MNIST(root="mnist", train=True, transform=transform, download=True)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(1):
    pbar = tqdm(train_loader, desc="Epoch 1", leave=False)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        pbar.set_postfix(loss=loss.item())

print("   MLP trained for 1 epoch – loss should be < 0.5 on last batches\n")

print("All foundation exercises complete!")


[4] Training a 2‑layer MLP on MNIST


   MLP trained for 1 epoch – loss should be < 0.5 on last batches

All foundation exercises complete!


In [18]:

print("\n Beginner knowledge complete.")


 Beginner knowledge complete.


Thanks for following